# LLM Preview Workbench

Notebook isolé pour tester une réponse LLM au-dessus de la branche retrieval `phase8b`.

Ce notebook montre:
- la requête brute
- le stage 1 documentaire
- les chunks envoyés au LLM
- le prompt citations-first
- la réponse LLM si `OPENAI_API_KEY` est disponible

Ce notebook n'est **pas** une promotion produit.
Il sert à auditer un preview end-to-end en mode contrôlé.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from bofip_cleanroom.env_utils import load_default_env_files
from bofip_cleanroom.preview_runtime import Phase8bPreviewRuntime
from bofip_cleanroom.llm_preview import (
    build_citation_prompt,
    generate_preview_answer,
    has_api_key,
)

_ = load_default_env_files()


In [ ]:
QUERY = "Notre startup a le statut JEI et porte des travaux de recherche. Peut-elle recuperer sa creance de CIR tout de suite ?"
RUN_LLM = False
PROVIDER = "gemini"
MODEL = "gemini-2.5-flash"


In [ ]:
runtime = Phase8bPreviewRuntime.from_local_corpus(corpus="commentary", device="cpu")
retrieval = runtime.retrieve(QUERY, top_docs=5, chunks_per_doc=3, max_chunks=8)

{
    "query": retrieval.query,
    "lexical_query": retrieval.lexical_query,
    "acronym_expansions": retrieval.acronym_expansions,
    "source_confidences": retrieval.source_confidences,
}


## Stage 1 documents


In [ ]:
[
    {
        "rank": hit.rank,
        "score": round(hit.score, 6),
        "boi_reference": hit.boi_reference,
        "title": hit.title,
        "sources": hit.sources,
        "ranks": hit.ranks,
    }
    for hit in retrieval.stage1_hits
]


## Stage 2 chunks envoyés au LLM


In [ ]:
[
    {
        "citation_id": chunk.citation_id,
        "boi_reference": chunk.boi_reference,
        "title": chunk.title,
        "section_path": chunk.section_path,
        "chunk_kind": chunk.chunk_kind,
        "text": chunk.text[:1200],
    }
    for chunk in retrieval.stage2_chunks
]


## Prompt


In [ ]:
prompt_text = build_citation_prompt(retrieval)
print(prompt_text)


## Réponse LLM


In [ ]:
if RUN_LLM:
    preview = generate_preview_answer(retrieval, provider=PROVIDER, model=MODEL)
    print(
        {
            "api_called": preview.api_called,
            "provider": preview.provider,
            "model": preview.model,
            "contract_version": preview.contract_version,
            "answer_valid": preview.answer_validation["valid"],
        }
    )
    print()
    print(preview.answer_text)
else:
    print({"api_key_present": has_api_key(PROVIDER), "provider": PROVIDER, "run_llm": RUN_LLM})
    print("Passe RUN_LLM = True si la clé du provider est disponible.")
